# 🔬 Kaggle GPU: Multilingual & Code-Switched Fake News Detection
### End-to-End Training & Benchmarking on Kaggle (T4 x 2 / P100 GPU)

**Models Evaluated:**
1. `google/muril-base-cased` (MuRIL - 110M params)
2. `ai4bharat/indic-bert` (IndicBERT - 12M params)
3. `xlm-roberta-base` (Multilingual RoBERTa - 278M params)
4. `bert-base-uncased` (English Monolingual Baseline - 110M params)

**Important Kaggle Settings:**
- In the right sidebar: **Settings** -> **Accelerator** -> Select **GPU T4 x 2** or **GPU P100**
- In the right sidebar: **Settings** -> **Internet** -> Toggle **Internet ON** (Required to download Hugging Face models)

## Step 1: GPU Diagnostics & Setup Environment

In [ ]:
!nvidia-smi
!pip install --quiet --upgrade torch torchvision torchaudio
!pip install --quiet transformers datasets accelerate scikit-learn pandas numpy matplotlib seaborn onnxruntime

## Step 2: Auto-Detect Dataset in Kaggle Input
This automatically searches `/kaggle/input` for your CSV/JSON dataset file.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# Search /kaggle/input for any dataset files
input_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)
print("Found files in /kaggle/input:", input_files)

if input_files:
    dataset_path = input_files[0]
    print(f"\nLoading primary dataset from: {dataset_path}")
    df = pd.read_csv(dataset_path)
else:
    print("No file found in /kaggle/input. Using fallback demonstration dataset...")
    sample_records = [
        ("विश्व स्वास्थ्य संगठन ने कोरोना वायरस के लिए नई गाइडलाइन जारी की।", 0, "hindi"),
        ("सावधान! कल रात 12 बजे से पृथ्वी पर कॉस्मिक किरणें आएंगी मोबाइल बंद रखें।", 1, "hindi"),
        ("RBI ne repo rate 6.5% par unchanged rakhne ka decision liya hai.", 0, "hinglish"),
        ("Govt PM Yojna ke under sabhi ko 5000 rs de rahi hai, link pe click karo!", 1, "hinglish"),
        ("NASA confirmed detection of atmospheric water vapor on exoplanet K2-18b.", 0, "english"),
        ("Boiled garlic water cures pneumonia and prevents viral infections in 12 hours.", 1, "english")
    ] * 20
    df = pd.DataFrame(sample_records, columns=["text", "label", "language"])

print(f"Shape: {df.shape}")
display(df.head())

## Step 3: Column Standardization & Indic Preprocessing

In [ ]:
import re
import unicodedata

# Auto-detect column names
lower_cols = {col.lower(): col for col in df.columns}
text_col = None
for c in ["text", "news", "content", "statement", "headline", "title", "tweet", "article"]:
    if c in lower_cols:
        text_col = lower_cols[c]
        break

label_col = None
for c in ["label", "target", "fake", "class", "veracity", "is_fake", "category"]:
    if c in lower_cols:
        label_col = lower_cols[c]
        break

print(f"Using text column: '{text_col}', label column: '{label_col}'")

# Clean labels to binary 0 (REAL) and 1 (FAKE)
LABEL_MAP = {
    "fake": 1, "1": 1, 1: 1, 1.0: 1, True: 1, "false": 1, "misleading": 1, "rumor": 1,
    "real": 0, "0": 0, 0: 0, 0.0: 0, False: 0, "true": 0, "authentic": 0, "verified": 0
}

def map_label(val):
    if pd.isna(val): return np.nan
    if isinstance(val, (int, float, bool)) and val in LABEL_MAP: return LABEL_MAP[val]
    s = str(val).strip().lower()
    return LABEL_MAP.get(s, np.nan)

df["clean_label"] = df[label_col].apply(map_label)
df = df.dropna(subset=[text_col, "clean_label"]).reset_index(drop=True)
df["clean_label"] = df["clean_label"].astype(int)

# Regex entity masking
URL_PAT = re.compile(r"(?:https?://|www\.)[a-zA-Z0-9.\-_~:/?#\[\]@!$&'()*+,;=%]+")
PHONE_PAT = re.compile(r"(?:\+91[\-\s]?)?[6-9]\d{9}|\b\d{3}[-.\s]??\d{3}[-.\s]??\d{4}\b")
ZERO_WIDTH = re.compile(r"[\u200B\u200C\u200D\u200E\u200F\uFEFF]")
ELONGATED = re.compile(r"(.)\1{2,}")

def preprocess_indic(text):
    if not isinstance(text, str): return "", "unknown"
    text = unicodedata.normalize("NFC", text)
    text = ZERO_WIDTH.sub("", text)
    text = URL_PAT.sub("[URL]", text)
    text = PHONE_PAT.sub("[PHONE]", text)
    text = ELONGATED.sub(r"\1\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    # Script detection
    dev_len = len(re.findall(r"[\u0900-\u097F]", text))
    lat_len = len(re.findall(r"[a-zA-Z]", text))
    tot = dev_len + lat_len
    if tot == 0: script = "unknown"
    elif dev_len / tot >= 0.6: script = "hindi_devanagari"
    elif lat_len / tot >= 0.6:
        hinglish_words = {"hai", "hain", "ko", "se", "me", "par", "ke", "ki", "ka", "bhai", "karo", "nahi"}
        script = "hinglish_roman" if len(set(text.lower().split()).intersection(hinglish_words)) >= 2 else "english_latin"
    else: script = "code_mixed"
    return text, script

cleaned_texts, scripts = zip(*[preprocess_indic(t) for t in df[text_col]])
df["cleaned_text"] = cleaned_texts
df["detected_script"] = scripts

print(f"Processed {len(df)} samples.")
print("Class Balance:\n", df["clean_label"].value_counts())
print("Scripts:\n", df["detected_script"].value_counts())

## Step 4: Stratified 80/10/10 Train / Val / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df["clean_label"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["clean_label"])

print(f"Train samples: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

## Step 5: Fine-Tune Models (MuRIL, IndicBERT, XLM-R, BERT Baseline)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active PyTorch Device: {device}")

MODELS_TO_TRAIN = {
    "muril": {"hf_name": "google/muril-base-cased", "lr": 2e-5, "batch": 16},
    "indic-bert": {"hf_name": "l3cube-pune/hindi-bert", "lr": 3e-5, "batch": 16},
    "xlm-roberta": {"hf_name": "xlm-roberta-base", "lr": 1.5e-5, "batch": 16},
    "bert-base-uncased": {"hf_name": "bert-base-uncased", "lr": 2e-5, "batch": 16}
}

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(texts, truncation=True, padding="max_length", max_length=max_length, return_tensors="pt")
        self.labels = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "macro_f1": f1, "precision": prec, "recall": rec}

os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
os.makedirs("/kaggle/working/results", exist_ok=True)
benchmark_summary = {}

for key, config in MODELS_TO_TRAIN.items():
    model_name = config["hf_name"]
    print(f"\n{'='*25} Training: {key} ({model_name}) {'='*25}")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
    except Exception as err:
        print(f"[Notice] Could not load model '{model_name}': {err}. Skipping...")
        continue
    
    train_ds = TextDataset(train_df["cleaned_text"].tolist(), train_df["clean_label"].tolist(), tokenizer)
    val_ds = TextDataset(val_df["cleaned_text"].tolist(), val_df["clean_label"].tolist(), tokenizer)
    test_ds = TextDataset(test_df["cleaned_text"].tolist(), test_df["clean_label"].tolist(), tokenizer)
    
    training_args = TrainingArguments(
        output_dir=f"/kaggle/working/checkpoints/{key}",
        num_train_epochs=3,
        per_device_train_batch_size=config["batch"],
        learning_rate=config["lr"],
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        fp16=torch.cuda.is_available(),
        logging_steps=20,
        report_to="none"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics
    )
    
    trainer.train()
    
    # Test set evaluation
    test_preds = trainer.predict(test_ds)
    y_preds = np.argmax(test_preds.predictions, axis=1)
    y_trues = test_df["clean_label"].values
    
    acc = accuracy_score(y_trues, y_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(y_trues, y_preds, average="macro", zero_division=0)
    cm = confusion_matrix(y_trues, y_preds).tolist()
    
    benchmark_summary[key] = {
        "accuracy": round(float(acc), 4),
        "macro_f1": round(float(f1), 4),
        "precision": round(float(prec), 4),
        "recall": round(float(rec), 4),
        "confusion_matrix": cm
    }
    
    # Save model and tokenizer for backend inference
    save_dir = f"/kaggle/working/checkpoints/{key}/best_model"
    trainer.save_model(save_dir)
    tokenizer.save_pretrained(save_dir)
    print(f"Saved {key} checkpoint to {save_dir}")

## Step 6: Benchmark Results Comparison & Export

In [ ]:
import json
import matplotlib.pyplot as plt

results_df = pd.DataFrame(benchmark_summary).T
print("=== Model Benchmark Comparison Table ===")
display(results_df)

# Save JSON and CSV
with open("/kaggle/working/results/benchmark_results.json", "w") as f:
    json.dump(benchmark_summary, f, indent=2)
results_df.to_csv("/kaggle/working/results/benchmark_results.csv")

# Package checkpoints for local FastAPI backend
!cd /kaggle/working && zip -r /kaggle/working/multilingual_checkpoints.zip checkpoints/ results/
print("\nCheckpoints successfully packaged to: /kaggle/working/multilingual_checkpoints.zip")
print("You can now download this zip file directly from the Kaggle Output section!")